In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device)

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
/home/mardon/Documents/ai-homework/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [3]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [4]:
lora_config = LoraConfig(r=50, target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj'])

In [5]:
model = get_peft_model(model, lora_config)

In [6]:
model.print_trainable_parameters()

trainable params: 78,643,200 || all params: 3,899,722,752 || trainable%: 2.0166


In [7]:
import json
from datasets import Dataset

In [8]:
data = json.load(open('instruction-data.json'))
raw_ds = Dataset.from_list(data)

In [9]:
def apply_chat_template(example):
    user_text = example['instruction'] + '\n' + example['input']
    ai_text = example['output']
    tokenized = tokenizer.apply_chat_template([
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': ai_text}
    ], tokenize=False)

    encoded = tokenizer(tokenized, truncation=True, padding=True)
    
    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'labels': encoded['input_ids'].copy()
    }

In [10]:
train_ds = raw_ds.map(apply_chat_template, remove_columns=raw_ds.column_names)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [11]:
data_collator = DataCollatorForSeq2Seq(model=model, tokenizer=tokenizer)

In [12]:
args = TrainingArguments(
    output_dir=f'{MODEL_NAME}-finetuned',
    bf16=True,
    tf32=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    optim='adamw_torch',
    report_to='none',
    dataloader_num_workers=4,
)

In [13]:
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    data_collator=data_collator,
    args=args
)

In [14]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=54, training_loss=1.6695310804578993, metrics={'train_runtime': 112.8628, 'train_samples_per_second': 29.239, 'train_steps_per_second': 0.478, 'total_flos': 4045564703416320.0, 'train_loss': 1.6695310804578993, 'epoch': 3.0})

In [15]:
trainer.save_model(f'{MODEL_NAME}-finetuned/final_model')